# Chapter 19 Walkthrough: Statistical Visualization & Publication-Ready Figures

**Book Reference:** Chapter 19, Sections 19.1 – 19.4  
Run each cell top-to-bottom (`Shift + Enter`) alongside the textbook dissections. Every figure uses Matplotlib's Object-Oriented (`fig, ax`) API, colorblind-safe Okabe-Ito hues, and clean figure lifecycle management (`plt.close(...)`).

In [ ]:
# @title Step 0: Install Tutor & Load Spine Datasets
!pip install -q sharukh-tutor
from pathlib import Path
import tempfile
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sharukh_tutor as tutor

clinic = tutor.load_data("clinic_triage", as_pandas=True)
clinic["bmi"] = clinic["weight_kg"] / ((clinic["height_cm"] / 100) ** 2)
mch = tutor.load_data("mch_survey_sample", as_pandas=True)
epi = tutor.load_data("epi_surveillance", as_pandas=True)

print("clinic_triage:    ", clinic.shape)
print("mch_survey_sample:", mch.shape)
print("epi_surveillance: ", epi.shape)


## Section 19.3.1 — Matplotlib Object-Oriented API (`Figure` -> `Axes` -> Artists)

In [ ]:
OKABE_ITO = {
    "blue": "#0072B2",
    "orange": "#E69F00",
    "vermilion": "#D55E00",
    "green": "#009E73",
    "sky": "#56B4E9",
}

fig, ax = plt.subplots(figsize=(6.5, 4.0))
hb = clinic["hemoglobin_g_dl"].dropna()
ax.hist(hb, bins=16, color=OKABE_ITO["blue"], edgecolor="white")
ax.axvline(
    11.0,
    color=OKABE_ITO["vermilion"],
    linestyle="--",
    linewidth=2.0,
    label="WHO Cutoff (11.0 g/dL)",
)
ax.annotate(
    "Anemic tail (<11.0)",
    xy=(11.0, 12),
    xytext=(8.8, 16),
    arrowprops={"arrowstyle": "->", "color": OKABE_ITO["vermilion"]},
    fontsize=9,
)
ax.set_title("Clinic Triage Hemoglobin Distribution")
ax.set_xlabel("Hemoglobin (g/dL)")
ax.set_ylabel("Patient Count")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False)

print("Patches:", len(ax.patches), "| Lines:", len(ax.lines))
print("Title:  ", ax.get_title())
plt.close(fig)


## Section 19.3.2 — Okabe-Ito Palette: Sorted Bars & Grouped Scatter

In [ ]:
district_summary = (
    clinic.assign(is_anemic=clinic["hemoglobin_g_dl"] < 11.0)
    .groupby("district", as_index=False)["is_anemic"]
    .mean()
    .rename(columns={"is_anemic": "anemia_pct"})
)
district_summary["anemia_pct"] *= 100
ordered = district_summary.sort_values("anemia_pct", ascending=True)

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.barh(ordered["district"], ordered["anemia_pct"], color="#009E73")
ax.set_title("Anemia Prevalence by District (Sorted)")
ax.set_xlabel("Anemia Prevalence (%)")
ax.set_ylabel("District")
top_row = ordered.iloc[-1]
print(f"Top district: {top_row['district']} ({top_row['anemia_pct']:.1f}%)")
plt.close(fig)

fig, ax = plt.subplots(figsize=(6.5, 4.0))
priority_colors = {"Routine": "#0072B2", "Moderate": "#D55E00"}
for priority, sub in clinic.groupby("triage_priority"):
    ax.scatter(
        sub["bmi"],
        sub["systolic_bp"],
        color=priority_colors.get(priority, "#E69F00"),
        alpha=0.8,
        label=priority,
    )
ax.set_title("BMI vs. Systolic BP by Triage Priority")
ax.set_xlabel("BMI (kg/m^2)")
ax.set_ylabel("Systolic Blood Pressure (mmHg)")
ax.legend(title="Triage Priority")
print("Scatter groups plotted:", len(ax.collections))
plt.close(fig)


## Section 19.3.3 — Time-Series Seasonality & 4-Week Rolling Mean

In [ ]:
dhaka = (
    epi[epi["district"] == "Dhaka"]
    .sort_values(["year", "epi_week"])
    .reset_index(drop=True)
)
x_weeks = range(1, len(dhaka) + 1)
roll_4w = dhaka["dengue_cases"].rolling(4, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.plot(
    x_weeks, dhaka["dengue_cases"], color="#56B4E9", alpha=0.65,
    label="Weekly Cases",
)
ax.plot(
    x_weeks, roll_4w, color="#D55E00", linewidth=2.2,
    label="4-Week Rolling Mean",
)
ax.set_title("Dhaka Weekly Dengue Cases & 4-Week Trend (2024-2025)")
ax.set_xlabel("Sequential Epidemiological Week")
ax.set_ylabel("Reported Dengue Cases")
ax.legend(frameon=False)
print("Peak weekly Dhaka cases:", int(dhaka["dengue_cases"].max()))
print("Peak 4-week rolling mean:", round(float(roll_4w.max()), 1))
plt.close(fig)


## Section 19.3.4 — Multi-Panel Subplots (`1x2` MCH Wealth Equity)

In [ ]:
quintile_order = ["Poorest", "Poorer", "Middle", "Richer", "Richest"]
mch_eq = (
    mch.groupby("wealth_quintile")[["child_stunted", "maternal_anemia"]]
    .mean()
    .reindex(quintile_order)
    * 100
)

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.0))
axes[0].bar(mch_eq.index, mch_eq["child_stunted"], color="#0072B2")
axes[0].set_title("Child Stunting Prevalence")
axes[0].set_xlabel("Wealth Quintile")
axes[0].set_ylabel("Prevalence (%)")

axes[1].bar(mch_eq.index, mch_eq["maternal_anemia"], color="#D55E00")
axes[1].set_title("Maternal Anemia Prevalence")
axes[1].set_xlabel("Wealth Quintile")
axes[1].set_ylabel("Prevalence (%)")

fig.suptitle("MCH Health Inequities Across Wealth Quintiles")
fig.tight_layout()
print("Poorest vs Richest stunting (%):")
print(
    round(float(mch_eq.loc["Poorest", "child_stunted"]), 1),
    "vs",
    round(float(mch_eq.loc["Richest", "child_stunted"]), 1),
)
plt.close(fig)


## Section 19.3.5 — Seaborn Correlation Heatmap & 300-DPI Export

In [ ]:
num_cols = ["age_years", "bmi", "systolic_bp", "hemoglobin_g_dl"]
corr_mat = clinic[num_cols].corr()

fig, ax = plt.subplots(figsize=(5.8, 4.6))
sns.heatmap(
    corr_mat, annot=True, fmt=".2f", center=0, cmap="coolwarm", ax=ax
)
ax.set_title("Clinic Triage Feature Correlation Matrix")

with tempfile.TemporaryDirectory() as tmpdir:
    png_path = Path(tmpdir) / "corr_heatmap.png"
    pdf_path = Path(tmpdir) / "corr_heatmap.pdf"
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    fig.savefig(pdf_path, dpi=300, bbox_inches="tight")
    print("300-DPI PNG bytes:", png_path.stat().st_size)
    print("Vector PDF bytes: ", pdf_path.stat().st_size)

plt.close(fig)


## Section 19.3.6 — Break It On Purpose! (Visualization Error Anatomy)

In [ ]:
# Intentional Bug #1: ValueError — mismatched x and y lengths in ax.plot
fig, ax = plt.subplots()
try:
    weeks = [1, 2, 3, 4]
    cases = [18, 29, 44]
    ax.plot(weeks, cases)
finally:
    plt.close(fig)


In [ ]:
# Intentional Bug #2: KeyError — plotting a misspelled column name
fig, ax = plt.subplots()
try:
    ax.hist(clinic["hb_g_dl"])
finally:
    plt.close(fig)


In [ ]:
# Intentional Bug #3: ValueError — invalid colormap name in sns.heatmap
fig, ax = plt.subplots()
try:
    sns.heatmap(
        corr_mat, cmap=plt.get_cmap("hospital_rainbow"), ax=ax
    )
finally:
    plt.close(fig)


## Section 19.4 — Worked Problem-Structuring Blueprint: Reproducible 2-Panel Publication Pipeline

In [ ]:
def generate_outbreak_report(epi_df, district, out_dir, dpi=300):
    """Build a 2x1 outbreak figure, export PNG+PDF, and close safely."""
    sub = (
        epi_df[epi_df["district"] == district]
        .sort_values(["year", "epi_week"])
        .reset_index(drop=True)
    )
    x = range(1, len(sub) + 1)
    roll = sub["dengue_cases"].rolling(4, min_periods=1).mean()

    fig, axes = plt.subplots(2, 1, sharex=True, figsize=(8.0, 5.2))
    axes[0].plot(x, sub["dengue_cases"], color="#56B4E9", alpha=0.6,
                 label="Weekly Cases")
    axes[0].plot(x, roll, color="#D55E00", linewidth=2.2,
                 label="4-Week Rolling Mean")
    axes[0].set_title(f"Dengue Surveillance & Rainfall — {district}")
    axes[0].set_ylabel("Dengue Cases")
    axes[0].legend(frameon=False)

    axes[1].bar(x, sub["rainfall_mm"], color="#0072B2", alpha=0.8)
    axes[1].set_title(f"Weekly Rainfall — {district}")
    axes[1].set_xlabel("Sequential Epidemiological Week")
    axes[1].set_ylabel("Rainfall (mm)")
    fig.tight_layout()

    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)
    png_p = out / f"{district.lower()}_outbreak.png"
    pdf_p = out / f"{district.lower()}_outbreak.pdf"
    fig.savefig(png_p, dpi=dpi, bbox_inches="tight")
    fig.savefig(pdf_p, dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return {
        "png_bytes": int(png_p.stat().st_size),
        "pdf_bytes": int(pdf_p.stat().st_size),
    }

with tempfile.TemporaryDirectory() as tmpdir:
    report = generate_outbreak_report(epi, "Dhaka", tmpdir, dpi=300)
    print("Exported Dhaka report bundle:", report)

plt.close("all")
print("Open Matplotlib figures remaining:", len(plt.get_fignums()))
